In [3]:
!pip install -q sentence-transformers faiss-cpu

In [4]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

In [5]:
documents = [

"""
Diabetes symptoms include increased thirst, fatigue,
frequent urination, blurred vision, and weight loss.
Patients should monitor blood glucose regularly.
""",

"""
Chest pain accompanied by shortness of breath,
sweating, nausea, or dizziness may indicate
a cardiac emergency requiring immediate medical care.
""",

"""
Dehydration symptoms include dizziness, dry mouth,
low urine output, weakness, and confusion.
Hydration is essential for recovery.
""",

"""
Hospital readmission risk increases in elderly patients,
patients with chronic illness, and patients with
multiple prior admissions.
""",

"""
High blood pressure may increase the risk of heart disease,
stroke, and kidney complications if untreated.
"""
]

In [6]:
model = SentenceTransformer(
    'all-MiniLM-L6-v2'
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
document_embeddings = model.encode(documents)

print(document_embeddings.shape)

(5, 384)


In [8]:
dimension = document_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(
    np.array(document_embeddings)
)

print("Vector database created successfully")

Vector database created successfully


In [9]:
def retrieve_documents(query, top_k=2):

    query_embedding = model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    results = []

    for idx in indices[0]:
        results.append(documents[idx])

    return results

In [10]:
results = retrieve_documents(
    "What are signs of diabetes?"
)

for r in results:
    print(r)
    print("--------")


Diabetes symptoms include increased thirst, fatigue,
frequent urination, blurred vision, and weight loss.
Patients should monitor blood glucose regularly.

--------

Dehydration symptoms include dizziness, dry mouth,
low urine output, weakness, and confusion.
Hydration is essential for recovery.

--------


In [11]:
def rag_chatbot(question):

    retrieved_docs = retrieve_documents(question)

    response = "Based on medical knowledge:\n\n"

    for doc in retrieved_docs:

        response += doc + "\n"

    response += """

This information is educational only and not a substitute for professional medical advice.
"""

    return response

In [12]:
answer = rag_chatbot(
    "What are symptoms of dehydration?"
)

print(answer)

Based on medical knowledge:


Dehydration symptoms include dizziness, dry mouth,
low urine output, weakness, and confusion.
Hydration is essential for recovery.


Diabetes symptoms include increased thirst, fatigue,
frequent urination, blurred vision, and weight loss.
Patients should monitor blood glucose regularly.



This information is educational only and not a substitute for professional medical advice.



In [ ]:
while True:

    question = input("Patient: ")

    if question.lower() == "exit":

        print("MediSphere AI: Stay healthy.")
        break

    response = rag_chatbot(question)

    print("\nMediSphere AI:")
    print(response)
    print("\n")


MediSphere AI:
Based on medical knowledge:


Hospital readmission risk increases in elderly patients,
patients with chronic illness, and patients with
multiple prior admissions.


High blood pressure may increase the risk of heart disease,
stroke, and kidney complications if untreated.



This information is educational only and not a substitute for professional medical advice.




MediSphere AI:
Based on medical knowledge:


Hospital readmission risk increases in elderly patients,
patients with chronic illness, and patients with
multiple prior admissions.


High blood pressure may increase the risk of heart disease,
stroke, and kidney complications if untreated.



This information is educational only and not a substitute for professional medical advice.




MediSphere AI:
Based on medical knowledge:


Dehydration symptoms include dizziness, dry mouth,
low urine output, weakness, and confusion.
Hydration is essential for recovery.


Diabetes symptoms include increased thirst, fatigue

# Retrieval-Augmented Generation (RAG)

This module enables MediSphere AI to retrieve relevant healthcare information from medical knowledge documents before generating responses.

Benefits:
- Reduced hallucinations
- More trustworthy healthcare guidance
- Context-aware medical assistance
- Scalable hospital knowledge integration

This architecture reflects modern enterprise AI system design.